# RealMLP n_ens=24 full 5-fold (exp_046) — exp_032(n_ens15) arch 동일, n_ens만 24

fold0 스크린(exp_040 n_ens24=0.953381 vs exp_032 n_ens15 fold0=0.952482, +0.0009 but 단일fold 노이즈)  
→ full 5-fold로 확정. 비교: exp_032 full OOF **0.953504**. 판정=개별+스택 게이트.  
※ exp_044(TabM)와 **동시 실행 실측** 겸용.

In [ ]:
# 1) input 자동탐색 (마운트 비표준: /kaggle/input/{datasets,competitions}/...) — torch import 前
import sys, os, glob, subprocess
from pathlib import Path
print('/kaggle/input:', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'NONE')
c = glob.glob('/kaggle/input/**/src/config.py', recursive=True); assert c, 'src/config.py 못 찾음'
SRC_ROOT = str(Path(c[0]).parents[1]); print('SRC_ROOT:', SRC_ROOT)
cc = glob.glob('/kaggle/input/**/playground-series-s6e5', recursive=True); assert cc, '대회 폴더 못 찾음'
COMP = Path(cc[0]); print('COMP:', COMP)
ac = glob.glob('/kaggle/input/**/f1_strategy_dataset*.csv', recursive=True); assert ac, '증강 csv 못 찾음'
AUG = Path(ac[0]); print('AUG:', AUG)

In [ ]:
# 2) GPU 종류 감지(nvidia-smi, torch import 前) → 조건부 torch. 그 위에 프로젝트 deps.
GPU = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', GPU)
def pip(*a): subprocess.run([sys.executable,'-m','pip','install','-q',*a], check=True)
if 'P100' in GPU:
    print('P100(sm_60) → cu121 torch trio 재설치 (Kaggle 기본 torch 는 sm_70+ 만)')
    pip('torch==2.5.1','torchvision==0.20.1','torchaudio==2.5.1',
        '--index-url','https://download.pytorch.org/whl/cu121')
else:
    print('T4 등(sm_75+) → Kaggle 기본 torch 유지')
pip('pytabkit','hydra-core','python-dotenv')

In [ ]:
# 3) torch CUDA 실연산 검증 + import 체인 fast-fail
import torch
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPU', torch.cuda.get_device_name(0))
_x = torch.randn(256, 256, device='cuda'); _v = (_x @ _x).sum().item()
print('CUDA matmul OK')
sys.path.insert(0, SRC_ROOT)
from src import config
from src.train_realmlp import run
print('import OK:', config.__file__)

In [ ]:
# 4) 경로 override
config.TRAIN_PATH = COMP / 'train.csv'
config.TEST_PATH = COMP / 'test.csv'
config.SAMPLE_SUBMISSION_PATH = COMP / 'sample_submission.csv'
config.SOURCE_AUG_PATH = AUG
out = Path('/kaggle/working')
config.OOF_DIR = out / 'oof'; config.SUBMISSION_DIR = out / 'submissions'; config.LOG_DIR = out / 'logs'
import pandas as pd
_a = pd.read_csv(config.SOURCE_AUG_PATH); print('AUG shape:', _a.shape)
assert len(_a) == 101371, f'증강 행수 불일치: {len(_a)}'
assert config.TRAIN_PATH.exists(), f'train.csv 없음: {config.TRAIN_PATH}'

In [ ]:
# 5) n_ens=24 full 5-fold — exp_032 arch 동일, n_ens=24, max_folds=None
from omegaconf import OmegaConf
import time
CONF = Path(SRC_ROOT) / 'conf'
mc = OmegaConf.load(CONF / 'model' / 'realmlp.yaml')
mc.params.n_epochs = 64
mc.params['n_ens'] = 24
mc.params['hidden_sizes'] = [512, 256, 128]
mc.params['act'] = 'silu'
mc.params['plr_sigma'] = 2.33
mc.params['embedding_size'] = 6
cfg = OmegaConf.create({
    'exp_id': 'exp_046_rmlp_nens24_full',
    'notes': 'RealMLP n_ens=24 full 5-fold vs exp_032 n_ens15 OOF 0.953504',
    'use_wandb': False,
    'max_folds': None,
    'model': mc,
    'features': OmegaConf.load(CONF / 'features' / 'realmlp_fe_v2.yaml'),
    'augment': {'enabled': True, 'weight': 1.0},
})
print(OmegaConf.to_yaml(cfg))
t0=time.time(); result=run(cfg); print(result, f'\n총 {time.time()-t0:.0f}s')
print('cv_mean =', result.get('cv_mean'), '| fold_scores =', result.get('fold_scores'))
log_path = out / 'logs' / 'exp_046_rmlp_nens24_full.json'
if log_path.exists():
    import json
    with open(log_path) as f: print('OOF AUC(log) =', json.load(f).get('oof_auc'), '| exp_032=0.953504')
